In [ ]:
"""Cell 1 - accelerator preflight (reused from dbald-gpu-probe). Confirm a real GPU BEFORE
spending the session on a ~31 GB download; enable_gpu is a request, not a guarantee (Kaggle hands out
CPU when the account's concurrent-GPU limit is taken). Abort early and cheaply if there is no GPU."""
import json, shutil, subprocess, time
r = {'t': time.time(), 'nvidia_smi_on_path': shutil.which('nvidia-smi') is not None}
try:
    r['nvidia_smi'] = subprocess.check_output(['nvidia-smi'], text=True)
except Exception as e:
    r['nvidia_smi'] = f'FAILED: {type(e).__name__}: {e}'
import torch
r['torch'] = torch.__version__
r['cuda_available'] = torch.cuda.is_available()
if torch.cuda.is_available():
    r['device_name'] = torch.cuda.get_device_name(0)
    r['total_vram_gib'] = round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2)
print(json.dumps(r, indent=2))
if not torch.cuda.is_available():
    raise SystemExit('No GPU on this machine (enable_gpu ignored, most likely the concurrent-GPU '
                     'session limit is taken). Nothing was downloaded; re-run when a GPU is free.')


In [ ]:
"""Cell 2 - environment assembly (reused from dbald-research). Pinned deps, the UltraEdit
diffusers fork (which provides StableDiffusion3InstructPix2PixPipeline that our code imports), our
own repo at the exact probe commit, and the verified UltraEdit checkpoint. token=False throughout:
public access only, no credentials anywhere."""
import hashlib, json, os, shutil, subprocess, sys, time
from pathlib import Path
TMP = Path('/kaggle/temp/probe'); TMP.mkdir(parents=True, exist_ok=True)
FORK_REV = '1af5f0478d56d62cf6f35781a2ab63e03d656b2d'
MODEL = 'BleachNick/SD3_UltraEdit_w_mask'
MODEL_REVISION = 'e5beecb69a3ef1399718be2f99b61730b8bf0346'
REPO_URL = 'https://github.com/hashimamjad-dev/DBALD-CVPR2026-Reproduction.git'
REPO_COMMIT = '31e3b784743276ed87ef58689d25fb499a8776b0'
started = time.monotonic()

# 1) pinned deps, isolated (exact pins from dbald-research)
DEPS = TMP / 'dependencies'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--target', str(DEPS), '--no-deps',
                'transformers==4.56.2', 'accelerate==1.10.1', 'huggingface-hub==0.34.4',
                'tokenizers==0.22.2', 'sentencepiece==0.2.1', 'safetensors==0.6.2'],
               check=True, timeout=600)
sys.path.insert(0, str(DEPS))

# 2) UltraEdit diffusers fork -> the SD3 masked InstructPix2Pix pipeline
FORK = TMP / 'UltraEdit'
subprocess.run(['git', 'clone', '--filter=blob:none', '--no-checkout',
                'https://github.com/HaozheZhao/UltraEdit.git', str(FORK)], check=True, timeout=300)
subprocess.run(['git', '-C', str(FORK), 'checkout', FORK_REV], check=True, timeout=300)
assert subprocess.check_output(['git', '-C', str(FORK), 'rev-parse', 'HEAD'], text=True).strip() == FORK_REV
sys.path.insert(0, str(FORK / 'diffusers/src'))
os.environ['HF_XET_CHUNK_CACHE_SIZE_BYTES'] = '0'

# 3) our repo at the exact probe commit (public clone, no creds)
REPO = TMP / 'dbald'
subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True, timeout=600)
subprocess.run(['git', '-C', str(REPO), 'checkout', REPO_COMMIT], check=True, timeout=120)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == REPO_COMMIT

# 3b) upstream LaneATT (gitignored in our repo) -> third_party/LaneATT for the R34 heatmap adapter
LANEATT_REV = '2f8583ba14eccba05e6779668bc3a38bc751984a'
LANEATT_DIR = REPO / 'third_party/LaneATT'
LANEATT_DIR.parent.mkdir(parents=True, exist_ok=True)
subprocess.run(['git', 'clone', '--filter=blob:none', '--no-checkout',
                'https://github.com/lucastabelini/LaneATT.git', str(LANEATT_DIR)], check=True, timeout=300)
subprocess.run(['git', '-C', str(LANEATT_DIR), 'checkout', LANEATT_REV], check=True, timeout=300)
assert subprocess.check_output(['git', '-C', str(LANEATT_DIR), 'rev-parse', 'HEAD'], text=True).strip() == LANEATT_REV

# 4) verified UltraEdit checkpoint (public, token=False); import-sanity the pipeline resolves
from huggingface_hub import HfApi, snapshot_download
import diffusers
from diffusers import StableDiffusion3InstructPix2PixPipeline  # noqa: F401
info = HfApi(token=False).model_info(MODEL, revision=MODEL_REVISION, files_metadata=True)
required = sum(f.size or 0 for f in info.siblings)
free = shutil.disk_usage(TMP).free
if free < required + 10 * 1024**3:
    raise RuntimeError(f'Insufficient scratch: need {required/2**30:.1f}+10 GiB, have {free/2**30:.1f}')
SNAPSHOT = Path(snapshot_download(MODEL, revision=info.sha, cache_dir=TMP / 'cache',
                                  token=False, max_workers=2))
for entry in info.siblings:                                    # cheap byte-size verify (dbald-research)
    p = SNAPSHOT / entry.rfilename
    if p.stat().st_size != entry.size:
        raise ValueError('size mismatch: ' + entry.rfilename)
print(json.dumps(dict(diffusers_version=diffusers.__version__, diffusers_src=diffusers.__file__,
                      snapshot=str(SNAPSHOT), checkpoint_revision=info.sha, download_bytes=required,
                      repo_commit=REPO_COMMIT, assembly_seconds=round(time.monotonic() - started, 1)),
                 indent=2))


In [ ]:
"""Cell 3 - DBALD poisoned-set generation for the LaneATT/TuSimple row. For each attack, run the
repo's split-aware mix builder with the paper_target UltraEdit config (drop_t5=True, no offload, 50
steps = the DEFAULT flags, i.e. no --full-t5 / no --cpu-offload) and the R34 gradient heatmap over the
pinned train split. LIMIT>0 poisons only the first LIMIT ordered train frames (fast end-to-end
validation); LIMIT=0 is the full 10% (326 per attack). Train images + the R34 checkpoint come from the
attached full-run-bundle; the split JSON and the mix code come from the cloned repo."""
import json, os, subprocess, sys, time
from pathlib import Path
ATTACKS = ['LDA']
LIMIT = 0                                               # 0 = full split; >0 = validation subset

INPUT = Path('/kaggle/input')                                  # discover nesting-agnostically (the dataset mounts with an extra full-run-bundle/ level)
_tl = next(iter(sorted(INPUT.rglob('tusimple_train/labels.jsonl'))), None)
if _tl is None:
    import itertools
    tree = '\n'.join(str(p) for p in itertools.islice(sorted(INPUT.rglob('*')), 80))
    raise SystemExit('tusimple_train/labels.jsonl not found under /kaggle/input. First entries:\n' + tree)
TRAIN = _tl.parent
_ck = next(iter(sorted(INPUT.rglob('laneatt_r34_tusimple_model_0100.pt'))), None)
if _ck is None:
    raise SystemExit('R34 checkpoint laneatt_r34_tusimple_model_0100.pt not found under /kaggle/input.')
CKPT = _ck
print(f'[gen] TRAIN={TRAIN}', flush=True)
print(f'[gen] CKPT={CKPT}', flush=True)
SPLIT = REPO / 'results/table1_split_20261111/tusimple_split.json'
if not SPLIT.exists():
    raise SystemExit(f'pinned split missing at {SPLIT}')

env = os.environ.copy()
env['PYTHONPATH'] = os.pathsep.join([str(FORK / 'diffusers/src'), str(DEPS), str(REPO / 'src')])
env['HF_HUB_OFFLINE'] = '1'
env['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

summary = {'repo_commit': REPO_COMMIT, 'limit': LIMIT, 'config': 'paper_target', 'attacks': {}}
for attack in ATTACKS:
    out = Path(f'/kaggle/working/dbald_{attack.lower()}')
    cmd = [sys.executable, str(REPO / 'scripts/mix_poisoned_dataset.py'),
           '--root', str(TRAIN), '--labels', str(TRAIN / 'labels.jsonl'),
           '--split-file', str(SPLIT), '--split-part', 'train',
           '--attack', attack, '--method', 'dbald', '--rate', '0.10',
           '--render-trigger', 'ultraedit', '--trigger-kind', 'mud', '--ultraedit-device', 'cuda',
           '--ultraedit-checkpoint', str(SNAPSHOT),
           '--checkpoint', str(CKPT), '--backbone', 'resnet34', '--device', 'cuda',
           '--output', str(out)]
    if LIMIT > 0:
        cmd += ['--limit', str(LIMIT)]
    print('RUN:', ' '.join(cmd), flush=True)
    t0 = time.time()
    rc = subprocess.run(cmd, env=env).returncode
    manifest = out / 'manifest.json'
    summary['attacks'][attack] = dict(returncode=rc, seconds=round(time.time() - t0, 1),
                                      manifest=json.loads(manifest.read_text()) if manifest.exists() else None)
    print(f'[gen] {attack} rc={rc} in {summary["attacks"][attack]["seconds"]}s', flush=True)
    if rc != 0:
        print(f'[gen] {attack} FAILED (rc={rc}); see log above.', flush=True)

Path('/kaggle/working/generation_summary.json').write_text(json.dumps(summary, indent=2, default=str))
print(json.dumps(summary, indent=2, default=str))
